# Fase 7 - Resultados finales

Este notebook consolida los resultados principales del proyecto **El Factor Local** para el alcance 1930-2014.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")

clean_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "world_cup_matches_clean.csv")
supervised_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "supervised_dataset.csv")
clusters_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "world_cup_editions_clusters.csv")

with open(PROJECT_ROOT / "outputs" / "metrics" / "supervised_metrics.json", "r", encoding="utf-8") as file:
    supervised_metrics = json.load(file)

with open(PROJECT_ROOT / "outputs" / "metrics" / "clustering_metrics.json", "r", encoding="utf-8") as file:
    clustering_metrics = json.load(file)

print(clean_df.shape)
print(supervised_df.shape)
print(clusters_df.shape)

## Alcance de datos

El proyecto trabaja con partidos de Copa Mundial entre 1930 y 2014. Se excluyeron ediciones posteriores porque la fuente complementaria `WorldCupMatches.csv` disponible en el proyecto no contiene la columna `stage` para 2018 y 2022.

In [ ]:
pd.DataFrame({
    "dataset": ["Partidos limpios", "Dataset supervisado", "Dataset clustering"],
    "rows": [len(clean_df), len(supervised_df), len(clusters_df)],
    "columns": [clean_df.shape[1], supervised_df.shape[1], clusters_df.shape[1]],
})

## Pregunta 1: existe ventaja observable para el continente sede?

Descriptivamente, los equipos del mismo continente de la sede tuvieron mayor tasa de avance que los equipos de otras regiones. Este resultado no prueba causalidad, pero apoya incluir `is_host_region` como variable contextual.

In [ ]:
host_region_summary = (
    supervised_df.groupby("is_host_region")
    .agg(advance_rate=("advanced_group_stage", "mean"), teams=("advanced_group_stage", "size"))
    .reset_index()
)
host_region_summary["region_type"] = host_region_summary["is_host_region"].map({0: "Otra region", 1: "Region sede"})
host_region_summary.round(3)

## Pregunta 2: que variables influyen mas en el avance de fase?

La regresion logistica sugiere que la experiencia historica, algunas confederaciones y el indicador de region sede aportan informacion predictiva. La interpretacion debe ser cautelosa porque el dataset es pequeno y las observaciones no son completamente independientes.

In [ ]:
supervised_metrics

## Pregunta 3: que tipos de mundiales aparecen al agrupar ediciones?

El mejor `k` segun silueta fue 3. K-Means y clustering jerarquico coincidieron en todas las asignaciones. Los grupos encontrados separan ediciones tempranas atipicas, ediciones intermedias y ediciones modernas.

In [ ]:
clusters_df[["year", "cluster_kmeans", "cluster_hier", "avg_total_goals", "n_matches", "n_teams", "overall_advance_rate"]]

In [ ]:
cluster_profile = clusters_df.groupby("cluster_kmeans").agg(
    editions=("year", lambda values: list(values.astype(int))),
    avg_total_goals=("avg_total_goals", "mean"),
    avg_abs_goal_diff=("avg_abs_goal_diff", "mean"),
    n_matches=("n_matches", "mean"),
    n_teams=("n_teams", "mean"),
    overall_advance_rate=("overall_advance_rate", "mean"),
).reset_index()
cluster_profile.round(3)

## Pregunta 4: el modelo supera al baseline?

Si. La regresion logistica supera al baseline en accuracy, F1 y AUC. La mejora es moderada y la accuracy queda por debajo de la referencia orientativa de 0.65, por lo que el resultado debe presentarse como una mejora limitada pero defendible.

In [ ]:
pd.DataFrame([
    {
        "model": "Baseline",
        "accuracy": supervised_metrics["baseline_accuracy"],
        "precision": supervised_metrics["baseline_precision"],
        "recall": supervised_metrics["baseline_recall"],
        "f1_score": supervised_metrics["baseline_f1_score"],
        "auc_roc": supervised_metrics["baseline_auc_roc"],
    },
    {
        "model": "Regresion logistica",
        "accuracy": supervised_metrics["model_accuracy"],
        "precision": supervised_metrics["precision"],
        "recall": supervised_metrics["recall"],
        "f1_score": supervised_metrics["f1_score"],
        "auc_roc": supervised_metrics["auc_roc"],
    },
])

## Limitaciones

1. Un mismo equipo aparece en multiples mundiales, por lo que las observaciones no son completamente independientes.
2. El dataset de clustering tiene solo 20 ediciones en el alcance 1930-2014.
3. La normalizacion de selecciones historicas simplifica entidades que cambiaron politicamente o deportivamente.
4. Las features pre-torneo disponibles son limitadas y pueden ser debiles para selecciones con pocas apariciones previas.
5. El formato de la Copa Mundial cambio fuertemente entre 1930 y 2014, lo que introduce heterogeneidad historica.

## Conclusion general

El proyecto construyo un pipeline reproducible desde datos crudos hasta modelos supervisados y no supervisados. El modelo supervisado supera al baseline, aunque con desempeno moderado. El clustering identifica perfiles historicos interpretables de torneos: ediciones tempranas atipicas, ediciones intermedias y ediciones modernas. En conjunto, los resultados sugieren que el contexto regional y el rendimiento historico aportan informacion, pero no son suficientes para explicar completamente el avance deportivo.